In [ ]:
# pip install qcd-gym gymnasium stable-baselines3 torch numpy
import gymnasium as gym
import numpy as np
from gymnasium import Wrapper
from stable_baselines3 import PPO

In [ ]:
# ---------- 1) 单比特门的矩阵 ----------
def RX(phi):
    c = np.cos(phi/2.0)
    s = -1j*np.sin(phi/2.0)
    return np.array([[c, s],
                     [s, c]], dtype=np.complex128)

def RZ(phi):
    return np.array([[np.exp(-1j*phi/2.0), 0],
                     [0, np.exp(+1j*phi/2.0)]], dtype=np.complex128)

I2 = np.eye(2, dtype=np.complex128)

# ---------- 2) 把 qcd-gym 的动作 <o,q,c,phi> 转换成我们这边的单比特门 ----------
# 约定（来自 qcd-gym 文档/示例）：o=0 -> PhaseShift (单比特相位门 ~ RZ up to global phase)
#                               o=1 且 q==c -> RX(phi)
#                               o=2 -> Terminate
def one_qubit_gate_from_action(action):
    o, q, c, phi = action
    o = int(o); q = int(q); c = int(c); phi = float(phi)
    if o == 0 and q == c:
        # PhaseShift 与 RZ 只差全局相位；合成任务中可忽略全局相位
        return RZ(phi)
    if o == 1 and q == c:
        return RX(phi)
    if o == 2:
        return None  # 终止
    # 其他（比如 CNOT）在 1 qubit 下无意义，这里返回恒等
    return I2

# ---------- 3) 自定义奖励 Wrapper ----------
class CustomUnitaryReward1Q(Wrapper):
    """
    用你给定的 U_target 覆盖 UC-random 的奖励：episode 结束时
    r = 1 - atan(||U_target - U_built||_F) - depth_penalty
    只支持 1 qubit（2x2）目标。
    """
    def __init__(self, env, U_target, depth_penalty=0.02):
        super().__init__(env)
        U_target = np.asarray(U_target, dtype=np.complex128)
        assert U_target.shape == (2,2), "Only 1-qubit 2x2 target is supported in this wrapper."
        self.U_target = U_target
        self.depth_penalty = depth_penalty
        self.reset(None)

    def reset(self, seed=None, options=None):
        obs, info = self.env.reset(seed=seed, options=options)
        self.steps = 0
        self.U_built = I2.copy()   # 从恒等开始，右乘新门（注意顺序约定）
        self.terminated = False
        self.truncated = False
        return obs, info

    def step(self, action):
        self.steps += 1
        # 累乘单比特门
        G = one_qubit_gate_from_action(action)
        if G is not None:
            # 约定：后添加的门在右侧（或左侧）乘？我们采用 V = G * V 的“前乘”习惯，
            # 只要始终一致即可。若方向与 qcd 内部不同不影响我们自定义奖励。
            self.U_built = G @ self.U_built

        obs, _reward_env, terminated, truncated, info = self.env.step(action)
        self.terminated |= terminated
        self.truncated |= truncated

        # 只有在 episode 结束时给真正奖励；中间步给 0（或微小 shaping）
        if self.terminated or self.truncated:
            fro_diff = np.linalg.norm(self.U_target - self.U_built, ord='fro')
            reward_final = 1.0 - np.arctan(fro_diff) - self.depth_penalty * (self.steps / getattr(self.env, "max_depth", 1))
            reward = float(np.real(reward_final))
        else:
            reward = 0.0  # 中间步不给力，鼓励尽快终止

        return obs, reward, terminated, truncated, info

# ---------- 4) 目标矩阵（随便来个 Y 门） ----------
U_target = np.array([[0, -1j],
                     [1j,  0]], dtype=np.complex128)  # Pauli-Y

# ---------- 5) 创建基础环境 & 包装 ----------
base_env = gym.make(
    "CircuitDesigner-v0",
    max_qubits=1,
    max_depth=6,
    objective="UC-random",   # 用它做“电路构造”的外壳，但奖励换成我们自己的
    render_mode="text",
)
env = CustomUnitaryReward1Q(base_env, U_target, depth_penalty=0.02)

# ---------- 6) 训练 ----------
model = PPO("MlpPolicy", env, verbose=1, n_steps=1024, batch_size=256, learning_rate=3e-4, gamma=0.995)
model.learn(total_timesteps=30_000)

# ---------- 7) 测试并打印学到的门序列 ----------
obs, info = env.reset(seed=0)
done = False
actions = []
while not done:
    action, _ = model.predict(obs, deterministic=True)
    actions.append(action.copy())
    obs, r, term, trunc, info = env.step(action)
    done = term or trunc

print("\nEpisode reward (custom):", r)

def decode(a):
    o,q,c,phi = a
    o,q,c = int(o), int(q), int(c)
    phi = float(phi)
    if o==0 and q==c: return f"P/RZ({phi:+.3f}) on q={q}"
    if o==1 and q==c: return f"RX({phi:+.3f}) on q={q}"
    if o==2:         return "Terminate"
    return f"Op{o} q={q} c={c} φ={phi:+.3f}"

print("Learned sequence:")
for i,a in enumerate(actions,1):
    print(f" {i:02d}. {decode(a)}")

# 渲染文本电路（来自 qcd-gym 的渲染）
try:
    env.render()
except Exception as e:
    print("(render not available:", e, ")")

env.close()


Using cpu device
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.


c:\ProgramData\anaconda3\envs\rl-research\Lib\site-packages\gymnasium\core.py:297: UserWarning: WARN: env.max_depth to get variables from other wrappers is deprecated and will be removed in v1.0, to get this variable you can do `env.unwrapped.max_depth` for environment variables or `env.get_attr('max_depth')` that will search the reminding wrappers.
  logger.warn(


---------------------------------
| rollout/           |          |
|    ep_len_mean     | 5.75     |
|    ep_rew_mean     | -0.222   |
| time/              |          |
|    fps             | 1233     |
|    iterations      | 1        |
|    time_elapsed    | 0        |
|    total_timesteps | 1024     |
---------------------------------
------------------------------------------
| rollout/                |              |
|    ep_len_mean          | 5.62         |
|    ep_rew_mean          | -0.22        |
| time/                   |              |
|    fps                  | 1181         |
|    iterations           | 2            |
|    time_elapsed         | 1            |
|    total_timesteps      | 2048         |
| train/                  |              |
|    approx_kl            | 0.0060252026 |
|    clip_fraction        | 0.0663       |
|    clip_range           | 0.2          |
|    entropy_loss         | -5.67        |
|    explained_variance   | -0.218       |
|    learning_r